# Step 1 — Substation Name Reconciliation

Spatial-match v1 hand-curated substations against our Phase 1B OSM substations and replace generic OSM names (`sub_osm_51`) with proper NGCP-style descriptions (`Bacolod substation, Bacolod City, Negros Occidental`).

**Inputs:**
- [data/buses.csv](../data/buses.csv) — 53 v1 hand-curated buses (NGCP-style names + coordinates)
- [backend/data/processed/buses.csv](../backend/data/processed/buses.csv) — Phase 1B/1C output

**Output:** updated `buses.csv` with reconciled `name` column; reconciliation report saved alongside.

**Match rule:** for each v1 substation, find the nearest current substation within `MAX_MATCH_KM = 2`. If matched, copy v1's `description` into the bus's `name`. v1 entries without a match are reported so we can decide whether to add them in step 2.

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

PROC_DIR = Path('../backend/data/processed')
DATA_DIR = Path('../data')

WGS = 'EPSG:4326'
UTM = 'EPSG:32651'
MAX_MATCH_KM = 1.0  # max distance between v1 and OSM substation to count as match

In [2]:
v1 = pd.read_csv(DATA_DIR / 'buses.csv')
current = pd.read_csv(PROC_DIR / 'buses.csv')
print(f'v1 buses: {len(v1)} | current buses: {len(current)}')

# Restrict matching to real substations (not synth distribution, not tower, not virtual root)
current_subs = current[current['bus_type'].isin(['substation', 'substation_synth'])].copy()
print(f'Current substations (substation + substation_synth): {len(current_subs)}')

v1 buses: 53 | current buses: 2428
Current substations (substation + substation_synth): 105


## §1 — Build GeoDataFrames and match

In [3]:
v1_g = gpd.GeoDataFrame(
    v1.rename(columns={'name': 'v1_name', 'description': 'v1_description',
                       'island': 'v1_island', 'v_nom': 'v1_voltage_kv',
                       'bus_type': 'v1_bus_type'}),
    geometry=gpd.points_from_xy(v1['x'], v1['y']), crs=WGS,
).to_crs(UTM)

current_g = gpd.GeoDataFrame(
    current_subs[['bus_id', 'name', 'voltage_kv', 'province', 'island', 'bus_type']],
    geometry=gpd.points_from_xy(current_subs['lon'], current_subs['lat']), crs=WGS,
).to_crs(UTM)

matched = gpd.sjoin_nearest(
    v1_g, current_g,
    how='left', distance_col='match_dist_m',
    max_distance=MAX_MATCH_KM * 1000,
)

n_matched   = matched['bus_id'].notna().sum()
n_unmatched = matched['bus_id'].isna().sum()
print(f'v1 entries matched to a current substation within {MAX_MATCH_KM} km: {n_matched}')
print(f'v1 entries unmatched (will be candidates for step 2 import): {n_unmatched}')

v1 entries matched to a current substation within 1.0 km: 27
v1 entries unmatched (will be candidates for step 2 import): 26


## §2 — Inspect matches

Show match distance distribution and the per-row match table. If multiple v1 entries match the same OSM substation, we'll see it here.

In [4]:
match_report = matched[matched['bus_id'].notna()][[
    'v1_name', 'v1_description', 'v1_island', 'v1_voltage_kv', 'v1_bus_type',
    'bus_id', 'name', 'voltage_kv', 'island', 'match_dist_m'
]].sort_values('match_dist_m').reset_index(drop=True)
match_report['match_dist_m'] = match_report['match_dist_m'].round(0)
print('Match distance distribution (m):')
print(match_report['match_dist_m'].describe().round(1))
print()
print('All matches:')
print(match_report.to_string())

Match distance distribution (m):
count     27.0
mean     192.8
std      183.9
min       46.0
25%       82.0
50%      116.0
75%      202.0
max      747.0
Name: match_dist_m, dtype: float64

All matches:
         v1_name                                          v1_description v1_island  v1_voltage_kv v1_bus_type                                               bus_id                                          name  voltage_kv  island  match_dist_m
0      04STARITA                            Santa Rita substation, Samar     Samar            138  substation                          sub_bagolibas_substation_48                          Bagolibas Substation        69.0   Samar          46.0
1   04STARITATAP                        Santa Rita tap substation, Samar     Samar            138  substation                          sub_bagolibas_substation_48                          Bagolibas Substation        69.0   Samar          46.0
2       04MAASIN                       Maasin substation, Southern Le

In [5]:
# Detect collisions: multiple v1 names mapped onto same OSM bus
collisions = match_report.groupby('bus_id').size()
collisions = collisions[collisions > 1]
if len(collisions):
    print('Collisions (multiple v1 entries → same OSM bus):')
    for bid, n in collisions.items():
        rows = match_report[match_report['bus_id'] == bid]
        print(f'  {bid} (×{n}):')
        for _, r in rows.iterrows():
            print(f'    {r["v1_name"]:<12} {r["v1_description"]:<60} {r["match_dist_m"]:.0f} m')
else:
    print('No collisions.')

Collisions (multiple v1 entries → same OSM bus):
  sub_bagolibas_substation_48 (×2):
    04STARITA    Santa Rita substation, Samar                                 46 m
    04STARITATAP Santa Rita tap substation, Samar                             46 m
  sub_colon_substation_67 (×2):
    05COLON      Colon substation, Cebu City                                  102 m
    05KSPC       Kepco SPC power plant substation, Cebu City                  528 m
  sub_magdugo_substation_72 (×3):
    05TOLEDO     Toledo substation, Cebu (copper smelter area)                116 m
    05TOLBESS    Toledo BESS substation, Cebu                                 116 m
    05MAGDUGO    Magdugo substation, Cebu                                     123 m


## §3 — Unmatched v1 entries (candidates for step 2)

In [6]:
unmatched = matched[matched['bus_id'].isna()][[
    'v1_name', 'v1_description', 'v1_island', 'v1_voltage_kv', 'v1_bus_type'
]].sort_values(['v1_island', 'v1_name']).reset_index(drop=True)
print(f'Unmatched ({len(unmatched)}):')
print(unmatched.to_string())

Unmatched (26):
         v1_name                                                  v1_description v1_island  v1_voltage_kv v1_bus_type
0      07CORELLA                                       Corella substation, Bohol     Bohol            138  substation
1        07TAPAL                                         Tapal substation, Bohol     Bohol            138  substation
2         05CEBU                                 Cebu main substation, Cebu City      Cebu            138  substation
3    05DAANBNTAY                                  Daan Bantayan substation, Cebu      Cebu            230  substation
4   05DAANLUNSOD                                   Daan Lungsod substation, Cebu      Cebu            230  substation
5         05NAGA                                           Naga substation, Cebu      Cebu            138  substation
6       05THERMA                     Therma Visayas power plant substation, Cebu      Cebu            138   generator
7       08BVISTA                        

## §4 — Apply name updates to buses.csv

For collisions, keep the v1 entry with the smallest match distance. Other collided v1 entries are reported for step 2 — they likely represent paired/adjacent substations OSM merged.

In [7]:
# Best v1 → bus_id mapping: lowest match_dist_m wins per bus_id
best = match_report.sort_values('match_dist_m').drop_duplicates('bus_id', keep='first')
name_map = dict(zip(best['bus_id'], best['v1_description']))
v1_code_map = dict(zip(best['bus_id'], best['v1_name']))
v1_type_map = dict(zip(best['bus_id'], best['v1_bus_type']))

buses = pd.read_csv(PROC_DIR / 'buses.csv')
buses['name_pre_reconcile'] = buses['name']
buses['name'] = buses.apply(
    lambda r: name_map.get(r['bus_id'], r['name']), axis=1)

# Track which buses got reconciled — useful for the journal / for the eventual map
buses['v1_code'] = buses['bus_id'].map(v1_code_map)
buses['v1_bus_type'] = buses['bus_id'].map(v1_type_map)

n_renamed = (buses['name'] != buses['name_pre_reconcile']).sum()
print(f'Buses renamed: {n_renamed}')

# Sample of renames
renamed = buses[buses['name'] != buses['name_pre_reconcile']][
    ['bus_id', 'name_pre_reconcile', 'name', 'v1_code', 'v1_bus_type']
].head(15)
print()
print('Sample of renamed buses:')
print(renamed.to_string(index=False))

Buses renamed: 23

Sample of renamed buses:
                                             bus_id                           name_pre_reconcile                                                   name      v1_code v1_bus_type
                            sub_cadiz_substation_56                             Cadiz Substation                    Cadiz substation, Negros Occidental      06CADIZ  substation
                       sub_compostela_substation_15                        Compostela Substation                            Compostela substation, Cebu   05COMPSTLA  substation
                            sub_colon_substation_67                             Colon Substation                            Colon substation, Cebu City      05COLON  substation
                          sub_tabango_substation_57                           Tabango Substation                     Tabango substation, Northern Leyte    04TABANGO  substation
                                         sub_osm_13                    

In [8]:
# Drop the helper column before saving (keep v1_code and v1_bus_type — useful metadata)
buses_out = buses.drop(columns=['name_pre_reconcile'])
buses_out.to_csv(PROC_DIR / 'buses.csv', index=False)
print(f'Wrote {PROC_DIR / "buses.csv"} ({len(buses_out)} rows)')

# Save full reconciliation report for the journal
match_report.to_csv(PROC_DIR / 'v1_reconciliation_matched.csv', index=False)
unmatched.to_csv(PROC_DIR / 'v1_reconciliation_unmatched.csv', index=False)
print(f'Wrote v1_reconciliation_matched.csv ({len(match_report)} rows)')
print(f'Wrote v1_reconciliation_unmatched.csv ({len(unmatched)} rows)')

Wrote ../backend/data/processed/buses.csv (2428 rows)
Wrote v1_reconciliation_matched.csv (27 rows)
Wrote v1_reconciliation_unmatched.csv (26 rows)
